In [ ]:
from datasets import Features, Value, Sequence, load_dataset, DownloadConfig, Dataset, load_from_disk
from datasets.combine import concatenate_datasets
import uuid, re, json
import langdetect
import random
from tqdm import tqdm
import os

In [2]:
import json
import csv

# File I/O utilities
def load_jsonl_to_list(jsonl_file_path):
    data_list = []
    with open(jsonl_file_path, 'r') as file:
        for line in file:
            json_obj = json.loads(line)
            data_list.append(json_obj)
    return data_list

# Load dataset
def load_dataset_from_file(filename):
    #if the file is json
    if filename.endswith('.json'):
        with open(filename, 'r') as file:
            return json.load(file)
    elif filename.endswith('.jsonl'):
        return load_jsonl_to_list(filename)
    else:
        raise ValueError("Invalid file format. Please provide a .json or .jsonl file.")

# Save dataset
def save_dataset(data, filename, convert_to_jsonl=False):
    if convert_to_jsonl:
        with open(filename, 'w') as file:
            for obj in data:
                file.write(json.dumps(obj) + '\n')
    else:
        with open(filename, 'w') as file:
            json.dump(data, file, indent=2)
            
# Open a CSV file for writing
def convert_to_csv(data, outputfile_path):
    with open(outputfile_path, 'w', newline='') as csv_file:
        # Create a CSV writer
        writer = csv.writer(csv_file)

        # Write the header row
        writer.writerow(data[0].keys())

        # Write the data rows
        for row in data:
            writer.writerow(row.values())

In [3]:
def get_filepaths(directory, ext=".json"):
    return [
        os.path.join(directory, filename)
        for filename in os.listdir(directory)
        if filename.endswith(ext)
    ]

In [4]:
input_dir = os.path.join(os.environ.get("CHECKPOINTS"), "real-world-mcp-apis","smithery-ai")
# print(input_dir)

## Load Smithery-ai crawled data

In [13]:
smithery_ai_files = get_filepaths(directory=input_dir, ext=".json")

smithery_ai_apis = []
for i, path in enumerate(smithery_ai_files):
    # print(f"{i}: {item}")
    smithery_ai_api = load_dataset_from_file(path)
    smithery_ai_apis.append(smithery_ai_api)
    
print(f"Total APIs: {len(smithery_ai_apis)}")


Total APIs: 2120


## Schema
- Each file corresponds to a single MCP server. In the context of tool-calling SDG, an MCP server represents a cohesive set of tools related to the same use case.
- The `labels` key contains LLM-generated tags provided by `Mistral-Small-3.2-24B-Instruct-2506` that are useful to classify MCP servers.
- Tool information is located under the `metadata` key:
    - `remote_server_response`: Obtained via the Smithery.ai API. If the subkey `is_success` is `True`, it indicates that our connectivity test to the MCP server was successful.
    - `server_info_crawled`: Used when the connectivity test failed. This information was obtained via web crawling.

Note: We recommend retrieving tool information from `remote_server_response` when `is_success` is `True`, as API descriptions in crawled data may be incomplete (e.g., truncated with ellipses in HTML pages), and trajectories cannot be constructed via HTTP responses. If the connectivity test fails, fallback to `server_info_crawled`.
- `license` information may be found under both the `remote_server_response` and/or `server_info_crawled` keys.
- `tools` list may be found under `remote_server_response` and/or `server_info_crawled` keys.

Files @s3 bucket: `.../tool-call-data/real-world-mcp-apis/smithery-ai/` `cf` prefix means connectivity test fail for that MCP server (file).

In [12]:
# Schema example
smithery_ai_apis[0]

{'labels': {'analysis': "The MCP Server focuses on interfacing with Datadog's API to provide monitoring, observability, and log management capabilities. Its primary functions include retrieving monitors, dashboards, metrics, events, and incidents, as well as searching and aggregating logs. This makes it a powerful tool for system monitoring, troubleshooting, and gaining insights into application performance.",
  'reasoning': 'The primary functionality revolves around monitoring and observability, which aligns closely with the predefined category "Monitoring & Observability" (or closest pre-defined category). Additional relevant labels include "Data Analysis & Processing" due to log aggregation and metric querying, and "Operating System" since it interacts with system-level metrics and events.',
  'primary_label': 'Others',
  'secondary_labels': ['Data Analysis & Processing', 'Operating System'],
  'custom_label': 'Monitoring & Observability\n     Note: "Monitoring & Observability" is a

### Classify by connectivity

In [9]:
from concurrent.futures import ThreadPoolExecutor

def is_successful(item):
    return item.get("metadata", {}).get("remote_server_response", {}).get("is_success", False)

def separate_by_connection_status(data, max_workers=8):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(is_successful, data))
    
    success_items = [item for item, success in zip(data, results) if success]
    failed_items  = [item for item, success in zip(data, results) if not success]
    
    return success_items, failed_items

smithery_ai_apis_conn_ok, smithery_ai_apis_conn_fail = separate_by_connection_status(smithery_ai_apis, max_workers=48)

print(f"✅ APIs with connectivity: {len(smithery_ai_apis_conn_ok)}")
print(f"❌ APIs without connectivity: {len(smithery_ai_apis_conn_fail)}")
print(f"📦 Total APIs: {len(smithery_ai_apis)}")


✅ APIs with connectivity: 871
❌ APIs without connectivity: 1249
📦 Total APIs: 2120


### Classify by main topic (llm-based annotation)

In [14]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

def count_primary_labels_chunk(chunk):
    counter = Counter()
    for item in chunk:
        label = item.get("labels", {}).get("primary_label")
        if label:
            counter[label] += 1
    return counter

def count_by_primary_label_parallel(data, num_workers=8):
    chunk_size = (len(data) + num_workers - 1) // num_workers
    chunks = [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(count_primary_labels_chunk, chunks))

    # Merge counters
    total = Counter()
    for c in results:
        total.update(c)

    return total

label_counts = count_by_primary_label_parallel(smithery_ai_apis, num_workers=48)

print("📊 Stats by primary_label:")
for label, count in label_counts.most_common():
    print(f"- {label}: {count}")


📊 Stats by primary_label:
- Web Search & Research: 247
- Development Tools: 213
- Database Operations: 169
- API Integration: 127
- Content Creation: 126
- Data Analysis & Processing: 109
- Cryptocurrency & Blockchain: 100
- Daily Productivity: 88
- AI/ML Tools: 87
- Weather: 77
- Communication Tools: 74
- Browser Automation: 72
- Financial Services: 71
- Operating System: 66
- Memory Management: 61
- Security & Authentication: 61
- Others: 59
- File Management: 47
- Travel & Maps: 44
- Social Media: 43
- Education: 31
- Gaming: 29
- Cloud Services: 24
- News & Media: 22
- Time & Calendar: 20
- Project Management: 20
- E-commerce: 19
- Health & Fitness: 14
